In [4]:
# ============================================================
# APEXPLANET - DATA ANALYTICS INTERNSHIP
# TASK 1: DATA IMMERSION & WRANGLING
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import os


# ------------------------------------------------------------
# 2. FILE PATHS
# ------------------------------------------------------------

INPUT_FILE = "ApexPlanet_DataAnalytics_Dataset.xlsx"

OUTPUT_DIR = "output"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


# ------------------------------------------------------------
# 3. LOAD DATASET
# ------------------------------------------------------------

print("=" * 70)
print("APEXPLANET - TASK 1: DATA IMMERSION & WRANGLING")
print("=" * 70)

print("\nLoading dataset...")

df = pd.read_excel(INPUT_FILE)

print("Dataset loaded successfully!")


# ------------------------------------------------------------
# 4. BASIC DATASET INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. DATASET INFORMATION")
print("=" * 70)

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for column in df.columns:
    print("-", column)

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())


# ------------------------------------------------------------
# 5. CREATE DATA DICTIONARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. DATA DICTIONARY")
print("=" * 70)

data_dictionary = pd.DataFrame({
    "Column_Name": [
        "Order_ID",
        "Order_Date",
        "Customer_ID",
        "Customer_Name",
        "Age",
        "Gender",
        "City",
        "Product",
        "Category",
        "Quantity",
        "Unit_Price",
        "Total_Sales"
    ],

    "Data_Type": [
        "String",
        "Date",
        "String",
        "String",
        "Numeric",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Integer",
        "Float",
        "Float"
    ],

    "Description": [
        "Unique identifier assigned to each order",
        "Date on which the order was placed",
        "Unique identifier assigned to each customer",
        "Name of the customer",
        "Age of the customer",
        "Gender of the customer",
        "City where the customer is located",
        "Product purchased by the customer",
        "Category of the purchased product",
        "Number of units purchased",
        "Price of one unit of the product",
        "Total sales value of the transaction"
    ],

    "Business_Relevance": [
        "Used to identify and track individual orders",
        "Useful for time-based sales analysis",
        "Used for customer-level analysis",
        "Useful for identifying customers",
        "Useful for demographic and customer segmentation",
        "Useful for demographic analysis",
        "Useful for geographical sales analysis",
        "Useful for product performance analysis",
        "Useful for category-level sales analysis",
        "Used to measure sales volume",
        "Used to calculate and analyze revenue",
        "Main sales/revenue measure for the transaction"
    ]
})

print(data_dictionary.to_string(index=False))

data_dictionary.to_excel(
    os.path.join(OUTPUT_DIR, "data_dictionary.xlsx"),
    index=False
)


# ------------------------------------------------------------
# 6. DATA QUALITY ASSESSMENT - MISSING VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. MISSING VALUE ANALYSIS")
print("=" * 70)

missing_values = df.isnull().sum()

print("\nMissing values in each column:")
print(missing_values)

print("\nTotal missing cells:", missing_values.sum())


# ------------------------------------------------------------
# 7. DATA QUALITY ASSESSMENT - DUPLICATE ROWS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. DUPLICATE ROW ANALYSIS")
print("=" * 70)

duplicate_rows = df.duplicated().sum()

print("\nCompletely duplicated rows:", duplicate_rows)


# ------------------------------------------------------------
# 8. ORDER_ID DUPLICATE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. ORDER ID ANALYSIS")
print("=" * 70)

order_id_duplicates = df[
    df["Order_ID"].duplicated(keep=False)
].sort_values("Order_ID")

print("\nDuplicate Order IDs:")

if len(order_id_duplicates) > 0:
    print(order_id_duplicates[
        ["Order_ID", "Order_Date", "Customer_ID"]
    ].to_string(index=False))
else:
    print("No duplicate Order IDs found.")


print("\nNumber of unique Order IDs:",
      df["Order_ID"].nunique())

print("Total records:",
      len(df))


# ------------------------------------------------------------
# 9. CHECK ORDER ID SEQUENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. ORDER ID SEQUENCE CHECK")
print("=" * 70)

# The dataset starts from ORD100002.
# Generate the expected sequence based on row position.

expected_order_ids = [
    "ORD" + str(100002 + i)
    for i in range(len(df))
]

df["Expected_Order_ID"] = expected_order_ids

order_id_errors = df[
    df["Order_ID"] != df["Expected_Order_ID"]
]

print("\nNumber of Order ID inconsistencies:",
      len(order_id_errors))

if len(order_id_errors) > 0:

    print("\nOrder ID inconsistencies found:")

    print(
        order_id_errors[
            ["Order_ID", "Expected_Order_ID", "Customer_ID"]
        ].to_string(index=True)
    )


# ------------------------------------------------------------
# 10. DATA TYPE CLEANING
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. DATA TYPE STANDARDIZATION")
print("=" * 70)

# Convert Order_Date to datetime
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

# Convert numerical columns
df["Age"] = pd.to_numeric(
    df["Age"],
    errors="coerce"
)

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

df["Unit_Price"] = pd.to_numeric(
    df["Unit_Price"],
    errors="coerce"
)

df["Total_Sales"] = pd.to_numeric(
    df["Total_Sales"],
    errors="coerce"
)

print("\nData types after standardization:")
print(df.dtypes)


# ------------------------------------------------------------
# 11. CLEAN TEXT COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. TEXT STANDARDIZATION")
print("=" * 70)

text_columns = [
    "Order_ID",
    "Customer_ID",
    "Customer_Name",
    "Gender",
    "City",
    "Product",
    "Category"
]

for column in text_columns:

    df[column] = df[column].astype("string")

    df[column] = df[column].str.strip()


print("\nText columns standardized successfully.")


# ------------------------------------------------------------
# 12. FIX CORRUPTED ORDER IDs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("9. ORDER ID CORRECTION")
print("=" * 70)

# The dataset contains 8 corrupted Order IDs.
# They appear as ORD100050 but their row positions indicate
# that they should be ORD100120, ORD100240, etc.
#
# We use the expected sequential Order ID identified earlier.

corrupted_count = (
    df["Order_ID"] != df["Expected_Order_ID"]
).sum()

print("\nCorrupted Order IDs detected:", corrupted_count)

# Replace incorrect IDs with the expected IDs

df.loc[
    df["Order_ID"] != df["Expected_Order_ID"],
    "Order_ID"
] = df.loc[
    df["Order_ID"] != df["Expected_Order_ID"],
    "Expected_Order_ID"
]

print("Order IDs corrected successfully.")


# Remove temporary column

df.drop(
    columns=["Expected_Order_ID"],
    inplace=True
)


# ------------------------------------------------------------
# 13. HANDLE MISSING AGE VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10. MISSING AGE HANDLING")
print("=" * 70)

missing_age_before = df["Age"].isnull().sum()

print("\nMissing Age values before cleaning:",
      missing_age_before)

# Use median age because Age is numerical
# and median is less affected by extreme values.

median_age = df["Age"].median()

df["Age"] = df["Age"].fillna(median_age)

print("Median age used for imputation:",
      median_age)

print("Missing Age values after cleaning:",
      df["Age"].isnull().sum())


# ------------------------------------------------------------
# 14. HANDLE MISSING CITY VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("11. MISSING CITY HANDLING")
print("=" * 70)

missing_city_before = df["City"].isnull().sum()

print("\nMissing City values before cleaning:",
      missing_city_before)

# Do not invent a customer's city.
# Replace missing city values with "Unknown".

df["City"] = df["City"].fillna("Unknown")

print("Missing City values after cleaning:",
      df["City"].isnull().sum())


# ------------------------------------------------------------
# 15. CHECK INVALID AGE VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("12. AGE VALIDATION")
print("=" * 70)

invalid_age = df[
    (df["Age"] < 0) |
    (df["Age"] > 100)
]

print("\nInvalid age records:", len(invalid_age))

if len(invalid_age) > 0:
    print(invalid_age[["Customer_ID", "Age"]])
else:
    print("No invalid age values found.")


# ------------------------------------------------------------
# 16. CHECK INVALID QUANTITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("13. QUANTITY VALIDATION")
print("=" * 70)

invalid_quantity = df[
    df["Quantity"] <= 0
]

print("\nInvalid quantity records:",
      len(invalid_quantity))

if len(invalid_quantity) == 0:
    print("No invalid quantity values found.")


# ------------------------------------------------------------
# 17. CHECK INVALID UNIT PRICE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("14. UNIT PRICE VALIDATION")
print("=" * 70)

invalid_price = df[
    df["Unit_Price"] <= 0
]

print("\nInvalid Unit Price records:",
      len(invalid_price))

if len(invalid_price) == 0:
    print("No invalid Unit Price values found.")


# ------------------------------------------------------------
# 18. VERIFY TOTAL SALES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("15. TOTAL SALES VALIDATION")
print("=" * 70)

# Calculate expected sales

df["Calculated_Sales"] = (
    df["Quantity"] *
    df["Unit_Price"]
)

# Calculate difference

df["Sales_Difference"] = (
    df["Total_Sales"] -
    df["Calculated_Sales"]
)

# Allow a small floating point tolerance

sales_errors = df[
    df["Sales_Difference"].abs() > 0.01
]

print("\nSales calculation mismatches:",
      len(sales_errors))

if len(sales_errors) == 0:
    print("All Total_Sales values are correct.")


# Remove temporary validation columns

df.drop(
    columns=[
        "Calculated_Sales",
        "Sales_Difference"
    ],
    inplace=True
)


# ------------------------------------------------------------
# 19. STANDARDIZE CATEGORY VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("16. CATEGORY STANDARDIZATION")
print("=" * 70)

df["Category"] = df["Category"].str.title()

df["Product"] = df["Product"].str.title()

df["Gender"] = df["Gender"].str.title()

df["City"] = df["City"].str.title()

print("\nCategories:")
print(df["Category"].unique())

print("\nProducts:")
print(df["Product"].unique())

print("\nGender values:")
print(df["Gender"].unique())


# ------------------------------------------------------------
# 20. FEATURE ENGINEERING - YEAR
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("17. FEATURE ENGINEERING")
print("=" * 70)

df["Year"] = df["Order_Date"].dt.year


# ------------------------------------------------------------
# 21. FEATURE ENGINEERING - MONTH
# ------------------------------------------------------------

df["Month"] = df["Order_Date"].dt.month


# ------------------------------------------------------------
# 22. FEATURE ENGINEERING - MONTH NAME
# ------------------------------------------------------------

df["Month_Name"] = df["Order_Date"].dt.month_name()


# ------------------------------------------------------------
# 23. FEATURE ENGINEERING - QUARTER
# ------------------------------------------------------------

df["Quarter"] = (
    "Q" +
    df["Order_Date"].dt.quarter.astype(str)
)


# ------------------------------------------------------------
# 24. FEATURE ENGINEERING - AGE GROUP
# ------------------------------------------------------------

def create_age_group(age):

    if age <= 25:
        return "18-25"

    elif age <= 35:
        return "26-35"

    elif age <= 50:
        return "36-50"

    else:
        return "51-65"


df["Age_Group"] = df["Age"].apply(
    create_age_group
)


# ------------------------------------------------------------
# 25. ROUND FINANCIAL VALUES
# ------------------------------------------------------------

df["Unit_Price"] = df["Unit_Price"].round(2)

df["Total_Sales"] = df["Total_Sales"].round(2)


# ------------------------------------------------------------
# 26. REMOVE COMPLETELY DUPLICATED ROWS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("18. FINAL DUPLICATE CHECK")
print("=" * 70)

duplicates_before = df.duplicated().sum()

print("\nDuplicate rows before removal:",
      duplicates_before)

if duplicates_before > 0:

    df = df.drop_duplicates()

    print(
        "Duplicate rows removed:",
        duplicates_before
    )

else:

    print("No duplicate rows found.")


# ------------------------------------------------------------
# 27. FINAL ORDER ID CHECK
# ------------------------------------------------------------

duplicate_order_ids_after = df[
    df["Order_ID"].duplicated(keep=False)
]

print(
    "\nDuplicate Order IDs after correction:",
    len(duplicate_order_ids_after)
)


# ------------------------------------------------------------
# 28. FINAL MISSING VALUE CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("19. FINAL MISSING VALUE CHECK")
print("=" * 70)

final_missing = df.isnull().sum()

print(final_missing)

print(
    "\nTotal remaining missing values:",
    final_missing.sum()
)


# ------------------------------------------------------------
# 29. FINAL DATASET INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("20. FINAL DATASET")
print("=" * 70)

print("\nFinal number of rows:",
      df.shape[0])

print("Final number of columns:",
      df.shape[1])

print("\nFinal columns:")

for column in df.columns:
    print("-", column)


# ------------------------------------------------------------
# 30. DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("21. DESCRIPTIVE STATISTICS")
print("=" * 70)

print(
    df.describe(include="all").transpose()
)


# ------------------------------------------------------------
# 31. SAVE CLEANED CSV
# ------------------------------------------------------------

cleaned_csv = os.path.join(
    OUTPUT_DIR,
    "cleaned_sales_dataset.csv"
)

df.to_csv(
    cleaned_csv,
    index=False
)

print(
    "\nCleaned CSV saved to:",
    cleaned_csv
)


# ------------------------------------------------------------
# 32. SAVE CLEANED EXCEL
# ------------------------------------------------------------

cleaned_excel = os.path.join(
    OUTPUT_DIR,
    "cleaned_sales_dataset.xlsx"
)

df.to_excel(
    cleaned_excel,
    index=False
)

print(
    "Cleaned Excel saved to:",
    cleaned_excel
)


# ------------------------------------------------------------
# 33. SAVE DATA QUALITY REPORT
# ------------------------------------------------------------

quality_report = pd.DataFrame({

    "Data_Quality_Check": [
        "Total Rows",
        "Total Columns",
        "Missing Age Before Cleaning",
        "Missing City Before Cleaning",
        "Completely Duplicate Rows",
        "Corrupted Order IDs",
        "Invalid Age Values",
        "Invalid Quantity Values",
        "Invalid Unit Price Values",
        "Total Sales Calculation Errors",
        "Remaining Missing Values",
        "Duplicate Order IDs After Cleaning"
    ],

    "Result": [
        len(df),
        len(df.columns),
        missing_age_before,
        missing_city_before,
        duplicate_rows,
        corrupted_count,
        len(invalid_age),
        len(invalid_quantity),
        len(invalid_price),
        len(sales_errors),
        int(df.isnull().sum().sum()),
        len(duplicate_order_ids_after)
    ]
})

quality_report_file = os.path.join(
    OUTPUT_DIR,
    "data_quality_report.xlsx"
)

quality_report.to_excel(
    quality_report_file,
    index=False
)

print(
    "Data quality report saved to:",
    quality_report_file
)


# ------------------------------------------------------------
# 34. SAVE SUMMARY REPORT
# ------------------------------------------------------------

summary = pd.DataFrame({

    "Metric": [
        "Total Transactions",
        "Unique Customers",
        "Total Sales",
        "Average Sales per Transaction",
        "Minimum Transaction Value",
        "Maximum Transaction Value",
        "Average Customer Age",
        "Total Quantity Sold"
    ],

    "Value": [
        len(df),
        df["Customer_ID"].nunique(),
        round(df["Total_Sales"].sum(), 2),
        round(df["Total_Sales"].mean(), 2),
        round(df["Total_Sales"].min(), 2),
        round(df["Total_Sales"].max(), 2),
        round(df["Age"].mean(), 2),
        df["Quantity"].sum()
    ]
})

summary_file = os.path.join(
    OUTPUT_DIR,
    "dataset_summary.xlsx"
)

summary.to_excel(
    summary_file,
    index=False
)


# ------------------------------------------------------------
# 35. FINAL SUCCESS MESSAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nFiles generated inside the 'output' folder:")

print("1. data_dictionary.xlsx")
print("2. cleaned_sales_dataset.csv")
print("3. cleaned_sales_dataset.xlsx")
print("4. data_quality_report.xlsx")
print("5. dataset_summary.xlsx")

print("\nYour dataset is now analysis-ready.")
print("=" * 70)

APEXPLANET - TASK 1: DATA IMMERSION & WRANGLING

Loading dataset...
Dataset loaded successfully!

1. DATASET INFORMATION

Number of rows: 1000
Number of columns: 12

Column names:
- Order_ID
- Order_Date
- Customer_ID
- Customer_Name
- Age
- Gender
- City
- Product
- Category
- Quantity
- Unit_Price
- Total_Sales

Data types:
Order_ID          object
Order_Date        object
Customer_ID       object
Customer_Name     object
Age              float64
Gender            object
City              object
Product           object
Category          object
Quantity           int64
Unit_Price       float64
Total_Sales      float64
dtype: object

First 5 rows:
    Order_ID  Order_Date Customer_ID Customer_Name   Age  Gender       City  \
0  ORD100002  2025-02-25    CUST5529  Customer_227  30.0  Female  Bengaluru   
1  ORD100003  2025-10-14    CUST3127  Customer_182  63.0    Male  Bengaluru   
2  ORD100004  2025-05-13    CUST8887  Customer_487  62.0  Female  Bengaluru   
3  ORD100005  2025-12-02   